# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We'll walk step-by-step through loading structured metadata, getting familiar with the data schema, extracting record sets, and performing basic processing and visualization.

### Dataset Source
The dataset source is provided by its Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR² dataset metadata and explore basic information.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show dataset metadata
md = dataset.metadata
print('Dataset:', getattr(md, 'name', None))
print('Description:', getattr(md, 'description', None))
print('Published:', getattr(md, 'datePublished', None))

## 2. Data Overview
Let's review available record sets, their fields, and their `@id` values. These IDs will be used for referencing specific data components when extracting records.

In [ ]:
# List record set IDs available
print('Record Sets:')
for rs in dataset.metadata.record_sets:
    print(f"  @id: {rs['@id']}  name: {rs.get('name', '(no name)')}")
    print('    Fields:')
    for f in rs['fields']:
        print(f"      @id: {f['@id']}  label: {f.get('name', f['@id'])}  dataType: {f.get('dataType', '')}")

## 3. Data Extraction
Extract the records from a chosen record set. For demonstration, we'll pick the main protein-level record set (using its `@id`).

In [ ]:
# -- List all record set IDs available --
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Record Set IDs:', record_sets)

# -- Pick the main record set for analysis (typically the first/main one) --
if len(record_sets) == 0:
    raise ValueError('No record sets found in the dataset.')
main_record_set_id = record_sets[0]

# Load all records into DataFrames for all record sets
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the main record set
print('Fields for record set', main_record_set_id)
print(list(dataframes[main_record_set_id].columns))
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main data table:
- Select a numeric field (e.g., coverage, MW, or normalized abundance) by its `@id`
- Filter records by threshold
- Normalize the field
- Group by another categorical field if available
  
_Please adjust field `@id` as needed for your analysis._

In [ ]:
# --- Set up your chosen fields by their @id (refer to cell above for options) ---
# For demonstration, let's suppose the fields are named 'coverage_pct' and 'gene_symbol' in the dataframe.
# Replace these with the actual @id column names from the above output if different.

numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field (example: coverage_pct or MW)
for col in dataframes[main_record_set_id].columns:
    if 'cover' in col.lower() or 'pct' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    for col in dataframes[main_record_set_id].select_dtypes(include='number').columns:
        numeric_field_id = col
        break

# Try to find a grouping field: e.g., gene or class
for col in dataframes[main_record_set_id].columns:
    if 'gene' in col.lower() or 'class' in col.lower() or 'type' in col.lower():
        group_field_id = col
        break

print('Numeric field selected:', numeric_field_id)
print('Group-by field selected:', group_field_id)

# Filter records where numeric field > threshold (e.g., coverage > 10)
threshold = 10
df = dataframes[main_record_set_id]

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Convert to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized column '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by group_field
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by '{group_field_id}':")
        display(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize the distribution of your selected numeric field (e.g., Coverage %) and a grouped mean if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        grouped_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20) # Top 20
        sns.barplot(x=grouped_means.values, y=grouped_means.index, orient='h')
        plt.title(f"Top 20 groups by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
With `mlcroissant`, we:
- Inspected dataset structure programmatically via `@id` resolution
- Extracted protein-level mass spectrometry records into DataFrames
- Performed basic EDA and normalization by field ID
- Visualized key field distributions

This approach ensures reproducibility and easy adaptation to other Croissant-compliant datasets. Continue exploring by filtering, transforming, and visualizing other fields by their `@id`!